# Chapter 9 — Pseudo-bulk Differential Expression
## Bridging Single-Cell Resolution and Bulk RNA-seq

**Chapter 9 | Elements of Style in Creating Workflows for Biomedical Research**

---

In Chapter 3 we ran limma-voom on 24 bulk RNA-seq samples and found a modest DE signal
between eutopic endometrium in endometriosis patients (EuE) and healthy controls (Ctrl).
The bulk signal was diluted — a handful of genes, modest fold changes.

Here we ask: **which cell types drive that signal?**

### Why pseudo-bulk?

Single-cell DE methods that treat each cell as an independent observation are statistically
flawed — cells from the same donor are not independent (Squair et al. 2021, *Nature Methods*).
The correct approach is **pseudo-bulk**: aggregate raw counts per (cell type × donor), then
apply the same bulk DE tools you already know — limma-voom.

This is not a new analysis method. It is the same limma-voom from Chapter 3, applied at
cell-type resolution instead of whole-tissue resolution.

### Dataset

Tan et al. 2022, *Nature Cell Biology* — GSE179640  
108,497 cells · 18 cell types (`celltype`) · 14 donors (`PID`) · 4 tissue conditions

| obs column | meaning |
|---|---|
| `celltype` | 18 author-annotated cell types (finer than `celltype_main`) |
| `PID` | patient/donor ID — the statistical unit for pseudo-bulk |
| `sample_type_rename` | condition (Control, EuE, EcP, EcO — to be confirmed) |
| `stage` | disease/tissue stage (to be confirmed) |
| `phase` | cell cycle phase (G1, S, G2M) — NOT disease stage |

> Tan Y, Flynn WF, Huang Z, et al. *Nature Cell Biology* 2022;24:1192–1204. PMID: 35864314.

## Step 1 — Load libraries

In [1]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import scipy.sparse
import warnings
warnings.filterwarnings('ignore')

sc.settings.verbosity = 1

# set_figure_params removed in newer IPython — set directly via matplotlib
plt.rcParams.update({
    'figure.dpi':       100,
    'figure.facecolor': 'white',
    'figure.figsize':   (6, 5),
})

print('scanpy', sc.__version__)
print('anndata', ad.__version__)

scanpy 1.9.6
anndata 0.11.4


/Users/adeslatt/miniforge3/envs/nsforest/lib/python3.11/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/Users/adeslatt/miniforge3/envs/nsforest/lib/python3.11/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/Users/adeslatt/miniforge3/envs/nsforest/lib/python3.11/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/Users/adeslatt/miniforge3/envs/nsforest/lib/python3.11/site-packages/anndata/utils.py:434: FutureWarning: Importing read_mtx from `anndata` is deprecated. Import anndata.io.read_mtx instead.
  warnings.warn(msg, FutureWarning)
/Users/adeslatt/miniforge3/envs/nsforest/lib/python3.11/site-packages/anndata/util

## Step 2 — Load the JAX h5ad and explore obs columns

The h5ad is downloaded by the sc-nsforest-qc-nf pipeline during the CloudOS run.
Here we load it directly from the JAX Single Cell Portal URL.

On first run this downloads ~1 GB — subsequent runs use the cached file.

In [2]:
import urllib.request, os

H5AD_URL  = 'https://singlecell.jax.org/datasets/endometriosis/h5ad/endo-2022_global.h5ad'
H5AD_PATH = 'data/endo-2022_global.h5ad'
os.makedirs('data', exist_ok=True)

if not os.path.exists(H5AD_PATH):
    print('Downloading h5ad (~1 GB) from JAX Single Cell Portal...')
    urllib.request.urlretrieve(H5AD_URL, H5AD_PATH)
    print(f'  Saved to {H5AD_PATH}')
else:
    size_mb = os.path.getsize(H5AD_PATH) / 1e6
    print(f'Cached: {H5AD_PATH}  ({size_mb:.0f} MB)')

adata = sc.read_h5ad(H5AD_PATH)
print(f'\nLoaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes')
print(f'obs columns: {list(adata.obs.columns)}')
print(f'obsm keys:   {list(adata.obsm.keys())}')

  Saved to data/endo-2022_global.h5ad

Loaded: 108,497 cells × 26,636 genes
obs columns: ['sequencing_saturation', 'stage', 'n_genes_by_counts', 'total_counts', 'pct_counts_mitochondrial', 'pct_counts_hemoglobin', 'celltype', 'celltype_main', 'sample_type_rename', 'PID', 'G2M_score', 'phase']
obsm keys:   ['X_pca', 'X_pca_uncorrected', 'X_umap']


## Step 3 — Investigate key obs columns

Before any analysis, understand what the authors provided.
This is what scsilhouette prints at the start of every run.

In [3]:
# Columns that define the analysis
for col in ['celltype', 'celltype_main', 'sample_type_rename', 'stage', 'phase', 'PID']:
    if col in adata.obs.columns:
        vals = adata.obs[col].value_counts()
        print(f'\n── {col} ({adata.obs[col].dtype}, {adata.obs[col].nunique()} unique) ──')
        print(vals.to_string())
    else:
        print(f'\n{col}: NOT FOUND')


── celltype (category, 18 unique) ──
celltype
fibroblast 1            37143
epithelial 2            16467
T-lymphocytes           12553
endothelial             10883
NK cells                 7677
mural cells              5773
myeloid 1                4859
myeloid 3                3761
myeloid 2                3205
B-lymphocytes            1626
mesothelial              1371
cilliated epithelial      848
epithelial 1              559
plasma B-lymphocyte       426
fibroblast 3              425
fibroblast 2              375
mast cells                361
pDC                       185

── celltype_main (category, 5 unique) ──
celltype_main
stromal        43716
lymphocytes    22282
epithelial     19245
myeloid        12371
endothelial    10883

── sample_type_rename (category, 5 unique) ──
sample_type_rename
EuE     30176
EcO     24982
EcP     24715
EcPA    14731
Ctrl    13893

── stage (category, 4 unique) ──
stage
4    52512
0    44069
2     6610
3     5306

── phase (category, 3 unique) ─

In [4]:
# Cross-tab: how many cells per donor per condition?
# This tells us the replication structure for pseudo-bulk

# Adjust column names here if they differ from above
CELLTYPE_COL   = 'celltype'
DONOR_COL      = 'PID'
CONDITION_COL  = 'sample_type_rename'   # update if needed after Step 3
TISSUE_COL     = 'stage'               # update if needed after Step 3

print('Cells per donor per condition:')
crosstab = pd.crosstab(adata.obs[DONOR_COL], adata.obs[CONDITION_COL])
print(crosstab)
print(f'\nTotal donors: {adata.obs[DONOR_COL].nunique()}')
print(f'Total conditions: {adata.obs[CONDITION_COL].nunique()}')

Cells per donor per condition:
sample_type_rename  Ctrl   EuE   EcP  EcPA    EcO
PID                                              
E01                    0  4615  2520  2786      0
E02                    0  3903  4220  2390      0
E03                    0  3883  5325  2925      0
E04                    0  3068  2513  1929      0
E05                    0  2384  2917  3531      0
E06                    0  3966  1931     0      0
E07                    0  2680  3638     0   2195
E08                    0  2040     0     0      0
E09                    0  3637  1651  1170   4463
C01                 2886     0     0     0      0
C02                 6211     0     0     0      0
E10                    0     0     0     0  11997
E11                    0     0     0     0   6327
C03                 4796     0     0     0      0

Total donors: 14
Total conditions: 5


## Step 4 — Filter to eutopic endometrium

Chapter 3 used eutopic endometrium only: 5 Ctrl vs 7 EuE.
For a fair comparison, we filter to the same tissue here.

Update `EUTOPIC_CONDITIONS` below based on the values you see in Step 3.
Likely values: `['Control', 'EuE']` or `['normal', 'endometriosis']`

In [5]:
# ── UPDATE THESE AFTER READING STEP 3 OUTPUT ──────────────────────────────────
EUTOPIC_CONDITIONS = ['Control', 'EuE']   # values from sample_type_rename
CTRL_VALUE         = 'Control'            # the reference condition
CASE_VALUE         = 'EuE'               # the case condition
# ──────────────────────────────────────────────────────────────────────────────

mask = adata.obs[CONDITION_COL].isin(EUTOPIC_CONDITIONS)
adata_eu = adata[mask].copy()

print(f'Eutopic subset: {adata_eu.n_obs:,} cells')
print(f'\nCondition breakdown:')
print(adata_eu.obs[CONDITION_COL].value_counts().to_string())
print(f'\nDonors in eutopic subset: {sorted(adata_eu.obs[DONOR_COL].unique())}')
print(f'\nCell type counts in eutopic subset:')
print(adata_eu.obs[CELLTYPE_COL].value_counts().to_string())

Eutopic subset: 30,176 cells

Condition breakdown:
sample_type_rename
EuE    30176

Donors in eutopic subset: ['E01', 'E02', 'E03', 'E04', 'E05', 'E06', 'E07', 'E08', 'E09']

Cell type counts in eutopic subset:
celltype
fibroblast 1            10561
epithelial 2             7121
T-lymphocytes            3493
NK cells                 2785
endothelial              1379
myeloid 1                1174
myeloid 3                 805
myeloid 2                 596
mural cells               563
B-lymphocytes             557
cilliated epithelial      350
mesothelial               263
mast cells                144
plasma B-lymphocyte       126
epithelial 1              106
fibroblast 2               69
pDC                        63
fibroblast 3               21


## Step 5 — Check for raw counts

limma-voom requires **raw integer counts** — not normalized, not log-transformed.
Many h5ad files store normalized values in `.X` and raw counts in `.raw.X` or a layer.
We must use the raw counts for pseudo-bulk aggregation.

In [6]:
def check_counts_layer(adata, name):
    X = adata.X
    if scipy.sparse.issparse(X):
        sample = X[:100].toarray()
    else:
        sample = np.array(X[:100])
    is_integer = np.all(sample == sample.astype(int))
    min_val = sample.min()
    max_val = sample.max()
    print(f'{name}: integers={is_integer}, min={min_val:.2f}, max={max_val:.2f}')

check_counts_layer(adata_eu, 'adata.X')

if adata_eu.raw is not None:
    print(f'adata.raw exists: {adata_eu.raw.X.shape}')
    adata_raw_check = adata_eu.raw.to_adata()
    check_counts_layer(adata_raw_check, 'adata.raw.X')

print(f'\nLayers available: {list(adata_eu.layers.keys())}')
for layer in adata_eu.layers:
    adata_eu_copy = adata_eu.copy()
    adata_eu_copy.X = adata_eu.layers[layer]
    check_counts_layer(adata_eu_copy, f'layer: {layer}')

adata.X: integers=False, min=0.00, max=7.90

Layers available: ['normed', 'raw']
layer: normed: integers=False, min=0.00, max=2706.07
layer: raw: integers=True, min=0.00, max=2130.00


In [7]:
# Set the counts matrix to use for pseudo-bulk
# Update this based on Step 5 output:
#   - If adata.X is integers -> USE_RAW = False
#   - If adata.raw.X is integers -> USE_RAW = True
#   - If a layer has integers -> set COUNTS_LAYER = 'counts' (or the layer name)

USE_RAW      = False     # set True if raw counts are in adata.raw.X
COUNTS_LAYER = None      # set to layer name if raw counts are in a layer

if USE_RAW and adata_eu.raw is not None:
    adata_counts = adata_eu.raw.to_adata()
    print('Using adata.raw.X for counts')
elif COUNTS_LAYER and COUNTS_LAYER in adata_eu.layers:
    adata_counts = adata_eu.copy()
    adata_counts.X = adata_eu.layers[COUNTS_LAYER]
    print(f'Using layer "{COUNTS_LAYER}" for counts')
else:
    adata_counts = adata_eu
    print('Using adata.X directly for counts')

print(f'Count matrix shape: {adata_counts.X.shape}')

Using adata.X directly for counts
Count matrix shape: (30176, 26636)


## Step 6 — Pseudo-bulk aggregation

For each **(cell type × donor × condition)** combination:
- Sum the raw counts across all cells
- Record number of cells contributing (for quality filtering)

This produces a counts matrix of shape **(genes × pseudo-samples)** —
exactly like the bulk RNA-seq matrix in Chapter 3, but per cell type.

In [ ]:
def pseudobulk_aggregate(adata, celltype_col, donor_col, condition_col, min_cells=10):
    """
    Aggregate raw counts to pseudo-bulk per (celltype × donor × condition).
    
    Returns:
        counts_dict  : {celltype -> (genes × samples) DataFrame}
        meta_dict    : {celltype -> sample metadata DataFrame}
    """
    counts_dict = {}
    meta_dict   = {}

    celltypes = sorted(adata.obs[celltype_col].unique())
    gene_names = adata.var_names

    for ct in celltypes:
        ct_mask = adata.obs[celltype_col] == ct
        ct_adata = adata[ct_mask]

        groups = ct_adata.obs.groupby([donor_col, condition_col])

        cols, meta_rows = [], []
        for (donor, cond), idx in groups.groups.items():
            n_cells = len(idx)
            if n_cells < min_cells:
                continue   # skip donors with too few cells
            X_sub = ct_adata[idx].X
            if scipy.sparse.issparse(X_sub):
                summed = np.array(X_sub.sum(axis=0)).flatten()
            else:
                summed = np.array(X_sub.sum(axis=0)).flatten()
            col_name = f'{ct}__{donor}__{cond}'
            cols.append(pd.Series(summed, index=gene_names, name=col_name))
            meta_rows.append({'sample': col_name, 'celltype': ct,
                              'donor': donor, 'condition': cond, 'n_cells': n_cells})

        if len(cols) >= 4:   # need at least 4 pseudo-samples for DE
            counts_dict[ct] = pd.DataFrame(cols).T
            meta_dict[ct]   = pd.DataFrame(meta_rows).set_index('sample')

    return counts_dict, meta_dict


print('Aggregating pseudo-bulk counts...')
counts_dict, meta_dict = pseudobulk_aggregate(
    adata_counts,
    celltype_col   = CELLTYPE_COL,
    donor_col      = DONOR_COL,
    condition_col  = CONDITION_COL,
    min_cells      = 10
)

print(f'\nCell types with sufficient replication for DE ({len(counts_dict)}):')
for ct, df in sorted(counts_dict.items(), key=lambda x: -x[1].shape[1]):
    meta = meta_dict[ct]
    n_ctrl = (meta['condition'] == CTRL_VALUE).sum()
    n_case = (meta['condition'] == CASE_VALUE).sum()
    print(f'  {ct:30s}  {df.shape[1]:3d} pseudo-samples  ({CTRL_VALUE}={n_ctrl}, {CASE_VALUE}={n_case})')

## Step 7 — Export pseudo-bulk counts for limma-voom (R)

We write each cell type's count matrix and metadata to CSV,
then run limma-voom via rpy2 (R inside Python).

This is the same limma-voom workflow as Chapter 3 —
just called once per cell type instead of once on the whole tissue.

In [ ]:
import os
os.makedirs('pseudobulk', exist_ok=True)

for ct, counts_df in counts_dict.items():
    safe_name = ct.replace(' ', '_').replace('/', '_')
    counts_df.to_csv(f'pseudobulk/{safe_name}_counts.csv')
    meta_dict[ct].to_csv(f'pseudobulk/{safe_name}_meta.csv')

print(f'Exported {len(counts_dict)} cell types to pseudobulk/')
print('Files written:')
for f in sorted(os.listdir('pseudobulk'))[:10]:
    print(f'  pseudobulk/{f}')
if len(os.listdir('pseudobulk')) > 10:
    print(f'  ... and {len(os.listdir("pseudobulk")) - 10} more')

## Step 8 — Go deeper: per-type h5ads with author subtypes

Bill Flynn (co-author, Tan et al. 2022) confirmed that finer subtype annotations
are available in separate h5ad files — one per major cell type:

| File | Main type | Example subtypes |
|------|-----------|------------------|
| `endo-2022_lymphocyte.h5ad` | lymphocytes | NK1, NK2, NK3, pNK, CTL, T_EM, T_Reg, ... |
| `endo-2022_stromal.h5ad` | stromal | fibroblast subtypes, smooth muscle, ... |
| `endo-2022_epithelial.h5ad` | epithelial | epithelial subtypes |
| `endo-2022_myeloid.h5ad` | myeloid | macrophage, DC, monocyte subtypes |
| `endo-2022_endothelial.h5ad` | endothelial | endothelial subtypes |

The `subtypes` column in each file gives the finest author annotation.
Each file has its own UMAP coordinates.

**For Chapter 9 we focus on two compartments:**
- **Stromal** — where MMP/POSTN/ACTA2 signal from Chapter 3 bulk DE should localise
- **Lymphocyte** — the immunotolerance story from the paper title

In [ ]:
# Download per-type h5ads (choose which compartments you want)
# Each is ~200-400 MB; cached after first download

JAX_BASE = 'https://singlecell.jax.org/datasets/endometriosis/h5ad'

PER_TYPE_FILES = {
    'stromal':     f'{JAX_BASE}/endo-2022_stromal.h5ad',
    'lymphocyte':  f'{JAX_BASE}/endo-2022_lymphocyte.h5ad',
    'epithelial':  f'{JAX_BASE}/endo-2022_epithelial.h5ad',
    'myeloid':     f'{JAX_BASE}/endo-2022_myeloid.h5ad',
    'endothelial': f'{JAX_BASE}/endo-2022_endothelial.h5ad',
}

# Set to True for the compartments you want to download
DOWNLOAD = {
    'stromal':     True,
    'lymphocyte':  True,
    'epithelial':  False,
    'myeloid':     False,
    'endothelial': False,
}

for name, url in PER_TYPE_FILES.items():
    local = f'data/endo-2022_{name}.h5ad'
    if DOWNLOAD.get(name):
        if not os.path.exists(local):
            print(f'Downloading {name} h5ad...')
            urllib.request.urlretrieve(url, local)
            print(f'  Saved: {local}')
        else:
            size_mb = os.path.getsize(local) / 1e6
            print(f'Cached: {local}  ({size_mb:.0f} MB)')
    else:
        print(f'Skip:   {name}')

In [ ]:
# Explore subtypes in each downloaded compartment

subtype_adatas = {}

for name in PER_TYPE_FILES:
    local = f'data/endo-2022_{name}.h5ad'
    if not os.path.exists(local):
        continue
    a = sc.read_h5ad(local)
    subtype_adatas[name] = a
    print(f'\n── {name}: {a.n_obs:,} cells ──')
    print(f'obs columns: {list(a.obs.columns)}')
    if 'subtypes' in a.obs.columns:
        print(a.obs.groupby(['celltype_main', 'subtypes']).size().to_string())
    elif 'celltype' in a.obs.columns:
        print(a.obs['celltype'].value_counts().to_string())

## Step 9 — Pseudo-bulk aggregation at subtype level

Re-use the same `pseudobulk_aggregate` function from Step 6,
but now applied to each per-type h5ad using the `subtypes` column.

Results land in `pseudobulk/subtypes/{compartment}/`.

In [ ]:
SUBTYPE_COL = 'subtypes'   # author annotation column in per-type h5ads

for name, a in subtype_adatas.items():
    ann_col = SUBTYPE_COL if SUBTYPE_COL in a.obs.columns else CELLTYPE_COL
    cond_col = CONDITION_COL if CONDITION_COL in a.obs.columns else None

    if cond_col is None:
        print(f'SKIP {name}: condition column not found')
        continue

    # Filter to eutopic conditions
    mask = a.obs[cond_col].isin(EUTOPIC_CONDITIONS)
    a_eu = a[mask].copy()

    print(f'\n── {name}: {a_eu.n_obs:,} eutopic cells, annotating by "{ann_col}" ──')

    ct_dict, meta_dict_sub = pseudobulk_aggregate(
        a_eu,
        celltype_col   = ann_col,
        donor_col      = DONOR_COL,
        condition_col  = cond_col,
        min_cells      = 10,
    )

    out_dir = f'pseudobulk/subtypes/{name}'
    os.makedirs(out_dir, exist_ok=True)

    for ct, counts_df in ct_dict.items():
        safe = ct.replace(' ', '_').replace('/', '_').replace('$', '').replace('{', '').replace('}', '')
        counts_df.to_csv(f'{out_dir}/{safe}_counts.csv')
        meta_dict_sub[ct].to_csv(f'{out_dir}/{safe}_meta.csv')

    print(f'  Exported {len(ct_dict)} subtypes to {out_dir}/')
    for ct, df in sorted(ct_dict.items()):
        meta = meta_dict_sub[ct]
        n_ctrl = (meta['condition'] == CTRL_VALUE).sum()
        n_case = (meta['condition'] == CASE_VALUE).sum()
        print(f'    {ct:30s}  {df.shape[1]:3d} pseudo-samples  (Ctrl={n_ctrl}, EuE={n_case})')

---
## Handoff to R — open `limma_voom_de.ipynb`

This notebook has exported two sets of pseudo-bulk count matrices:

```
pseudobulk/
  {celltype}_counts.csv          ← global 18-type analysis
  {celltype}_meta.csv
  subtypes/
    stromal/
      {subtype}_counts.csv       ← stromal compartment (fibroblast subtypes etc.)
      {subtype}_meta.csv
    lymphocyte/
      {subtype}_counts.csv       ← immune compartment (NK, T cell subtypes etc.)
      {subtype}_meta.csv
```

**Next:** open `limma_voom_de.ipynb` — R kernel — which reads these CSVs
and runs limma-voom, producing volcano plots and the Chapter 3 cross-reference table.